Data Cleaning

In [ ]:
import pandas as pd
import re, os

BASE_DIR   = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
DATA_DIR  = os.path.join(BASE_DIR, 'data')

df = pd.read_csv(os.path.join(DATA_DIR, 'raw', 'combined_all_data.csv'))

print(f"Total awal: {len(df)}")
print("-" * 60)

# --- CEK PENYEBAB PER TAHAP ---

# 1. Duplikat
duplikat = df.duplicated().sum()
print(f"1. Duplikat                    : {duplikat} baris")

# 2. NaN per kolom
print(f"\n2. NaN per kolom:")
for col in df.columns:
    n_nan = df[col].isna().sum()
    pct   = n_nan / len(df) * 100
    print(f"   {col:<20}: {n_nan} baris ({pct:.1f}%)")

# 3. Teks kosong setelah cleaning
def clean_text_test(text):
    if pd.isna(text): return ""
    text = str(text)
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_test'] = df['text'].apply(clean_text_test)
teks_kosong = (df['text_test'].str.strip() == '').sum()
print(f"\n3. Teks kosong setelah cleaning: {teks_kosong} baris")

# 4. Timestamp invalid
ts_invalid = pd.to_datetime(df['timestamp'], errors='coerce').isna().sum()
print(f"4. Timestamp invalid            : {ts_invalid} baris")

# 5. Sampel teks yang akan jadi kosong
print(f"\n=== SAMPEL TEKS YANG AKAN TERHAPUS ===")
mask_kosong = df['text_test'].str.strip() == ''
print(df[mask_kosong]['text'].head(10).to_string())

Total awal: 81554
------------------------------------------------------------
1. Duplikat                    : 0 baris

2. NaN per kolom:
   id                  : 0 baris (0.0%)
   timestamp           : 0 baris (0.0%)
   ownerUsername       : 0 baris (0.0%)
   text                : 537 baris (0.7%)
   likesCount          : 0 baris (0.0%)
   postUrl             : 0 baris (0.0%)
   commentUrl          : 0 baris (0.0%)
   source_file         : 0 baris (0.0%)

3. Teks kosong setelah cleaning: 1391 baris
4. Timestamp invalid            : 37420 baris

=== SAMPEL TEKS YANG AKAN TERHAPUS ===
32                                   @dpr_ri
66     @prabowo @gibran_rakabuming @gerindra
70                                       NaN
75                      @muhammad_faridhatul
77                                       NaN
89                                       NaN
173                                      NaN
174                                      NaN
175                                      NaN
195

In [4]:
# Cek sampel timestamp yang invalid
# df_raw = pd.read_csv('01_data_scraping/combined_all_data.csv')
df_raw = df

mask_invalid = pd.to_datetime(df_raw['timestamp'], errors='coerce').isna()

print("=== SAMPEL TIMESTAMP INVALID ===")
print(df_raw[mask_invalid]['timestamp'].head(20).to_string())

print("\n=== SAMPEL SOURCE FILE DENGAN TIMESTAMP INVALID ===")
print(df_raw[mask_invalid]['source_file'].value_counts().head(10))

=== SAMPEL TIMESTAMP INVALID ===
25347    2026-02-25T20:51:09.000Z
25348    2026-02-25T22:08:26.000Z
25349    2026-02-28T11:33:09.000Z
25350    2026-02-25T18:25:59.000Z
25351    2026-02-26T02:26:24.000Z
25352    2026-02-28T09:53:07.000Z
25353    2026-02-26T13:12:01.000Z
25354    2026-03-01T04:16:58.000Z
25355    2026-03-01T03:33:49.000Z
25356    2026-02-26T22:44:37.000Z
25357    2026-02-26T21:29:11.000Z
25358    2026-03-01T07:44:31.000Z
25359    2026-03-02T14:55:54.000Z
25360    2026-02-26T07:03:37.000Z
25361    2026-02-26T09:47:30.000Z
25362    2026-02-27T07:08:17.000Z
25363    2026-02-28T12:34:31.000Z
25364    2026-02-26T22:12:30.000Z
25365    2026-03-01T04:16:46.000Z
25366    2026-02-27T02:42:48.000Z

=== SAMPEL SOURCE FILE DENGAN TIMESTAMP INVALID ===
source_file
dataset_tiktok_20260525_162117.csv    4000
dataset_tiktok_20260525_201049.csv    4000
dataset_tiktok_20260525_182805.csv    4000
dataset_tiktok_20260525_195849.csv    3999
dataset_tiktok_20260525_184348.csv    2751
dataset

In [5]:
# Cek format timestamp TikTok secara detail
# df_raw = pd.read_csv('combined_all_data.csv')

df_raw = df

mask_tiktok = df_raw['source_file'].str.contains('tiktok', case=False, na=False)

print("=== SAMPEL TIMESTAMP TIKTOK (RAW) ===")
print(df_raw[mask_tiktok]['timestamp'].head(10).to_string())
print(f"\nTipe data timestamp: {df_raw['timestamp'].dtype}")

# Coba parse manual berbagai format
sample = df_raw[mask_tiktok]['timestamp'].iloc[0]
print(f"\nContoh nilai: '{sample}'")
print(f"Panjang string: {len(str(sample))}")

# Coba berbagai format
import pandas as pd
formats = [
    '%Y-%m-%dT%H:%M:%S.%fZ',
    '%Y-%m-%dT%H:%M:%SZ',
    '%Y-%m-%d %H:%M:%S',
    '%d/%m/%Y %H:%M:%S',
]
for fmt in formats:
    try:
        hasil = pd.to_datetime(sample, format=fmt)
        print(f"✅ Format '{fmt}' BERHASIL: {hasil}")
    except:
        print(f"❌ Format '{fmt}' gagal")

=== SAMPEL TIMESTAMP TIKTOK (RAW) ===
25347    2026-02-25T20:51:09.000Z
25348    2026-02-25T22:08:26.000Z
25349    2026-02-28T11:33:09.000Z
25350    2026-02-25T18:25:59.000Z
25351    2026-02-26T02:26:24.000Z
25352    2026-02-28T09:53:07.000Z
25353    2026-02-26T13:12:01.000Z
25354    2026-03-01T04:16:58.000Z
25355    2026-03-01T03:33:49.000Z
25356    2026-02-26T22:44:37.000Z

Tipe data timestamp: object

Contoh nilai: '2026-02-25T20:51:09.000Z'
Panjang string: 24
✅ Format '%Y-%m-%dT%H:%M:%S.%fZ' BERHASIL: 2026-02-25 20:51:09
❌ Format '%Y-%m-%dT%H:%M:%SZ' gagal
❌ Format '%Y-%m-%d %H:%M:%S' gagal
❌ Format '%d/%m/%Y %H:%M:%S' gagal


In [6]:
import pandas as pd
import re
import os

BASE_DIR   = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
DATA_DIR  = os.path.join(BASE_DIR, 'data')

df = pd.read_csv(os.path.join(DATA_DIR, 'raw', 'combined_all_data.csv'))
print(f"Jumlah data sebelum cleaning: {len(df)}")
print("-" * 60)

# --- 1. HAPUS DUPLIKAT ---
sebelum = len(df)
df = df.drop_duplicates()
print(f"Duplikat dihapus             : {sebelum - len(df)} baris")

# --- 2. HAPUS NaN HANYA PADA KOLOM TEXT ---
sebelum = len(df)
df = df.dropna(subset=['text'])
print(f"NaN teks dihapus             : {sebelum - len(df)} baris")

# --- 3. PERBAIKAN TIMESTAMP — HANDLE 2 FORMAT ---
def parse_timestamp(ts):
    """
    Handle dua format timestamp:
    - Instagram : '2025-10-20 02:24:41+00:00'
    - TikTok    : '2026-02-25T20:51:09.000Z'
    """
    if pd.isna(ts):
        return pd.NaT
    ts = str(ts).strip()
    # Format TikTok
    try:
        return pd.to_datetime(ts, format='%Y-%m-%dT%H:%M:%S.%fZ', utc=True)
    except:
        pass
    # Format Instagram
    try:
        return pd.to_datetime(ts, utc=True)
    except:
        return pd.NaT

print("Memproses timestamp... (mohon tunggu)")
df['timestamp'] = df['timestamp'].apply(parse_timestamp)

sebelum = len(df)
df = df.dropna(subset=['timestamp'])
print(f"Timestamp invalid dihapus    : {sebelum - len(df)} baris")

# --- 4. TEXT CLEANING ---
def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)

# --- 5. BERSIHKAN KOLOM LAIN ---
df['ownerUsername'] = df['ownerUsername'].astype(str).apply(
    lambda x: re.sub(r'[^A-Za-z0-9_.]', '', x)
)
df['likesCount'] = pd.to_numeric(
    df['likesCount'], errors='coerce'
).fillna(0).astype(int)
df['postUrl']     = df['postUrl'].astype(str).apply(str.strip)
df['commentUrl']  = df['commentUrl'].astype(str).apply(str.strip)
df['source_file'] = df['source_file'].astype(str).apply(
    lambda x: re.sub(r'[^A-Za-z0-9_.-]', '', x)
)

# --- 6. HAPUS TEKS KOSONG SETELAH CLEANING ---
sebelum = len(df)
df = df[df['text'].str.strip() != '']
print(f"Teks kosong dihapus          : {sebelum - len(df)} baris")

# --- 7. LAPORAN FINAL ---
print("-" * 60)
print(f"Total data setelah cleaning  : {len(df)}")
print(f"\n=== DISTRIBUSI SOURCE FILE ===")
print(df['source_file'].value_counts().to_string())

# --- 8. SIMPAN ---
output_path = os.path.join(DATA_DIR,'interim', 'cleaned_data.csv')
df.to_csv(output_path, index=False)
print(f"\n✅ Data cleaned tersimpan di: {output_path}")

Jumlah data sebelum cleaning: 81554
------------------------------------------------------------
Duplikat dihapus             : 0 baris
NaN teks dihapus             : 537 baris
Memproses timestamp... (mohon tunggu)
Timestamp invalid dihapus    : 0 baris
Teks kosong dihapus          : 854 baris
------------------------------------------------------------
Total data setelah cleaning  : 80163

=== DISTRIBUSI SOURCE FILE ===
source_file
dataset_instagram_20260525_143730.csv    4697
dataset_tiktok_20260525_195849.csv       3992
dataset_tiktok_20260525_162117.csv       3971
dataset_tiktok_20260525_182805.csv       3965
dataset_instagram_20260525_150052.csv    3947
dataset_tiktok_20260525_201049.csv       3852
data_scrapping_10.22.2025_18.55.csv      3186
data_scrapping_10.22.2025_21.05.csv      2998
dataset_instagram_20260525_142425.csv    2951
dataset_tiktok_20260525_184348.csv       2736
dataset_tiktok_20260525_165715.csv       2448
dataset_instagram_20260525_140226.csv    2090
dataset_tik

In [7]:
# --- LIBRARIES ---
import pandas as pd
import re

# --- 1. DATA LOADING ---
# Membaca file CSV hasil scraping sebelumnya
df = pd.read_csv(os.path.join(DATA_DIR,'interim', 'cleaned_data.csv'))
print(f"Jumlah data sebelum cleaning: {df.shape}")

# --- 2. DEDUPLICATION & MISSING VALUES HANDLING ---
# Menghapus duplikasi baris data
df = df.drop_duplicates()
print(f"Setelah hapus duplikat: {df.shape}")

# Menghapus baris yang memiliki nilai kosong (NaN)
df = df.dropna()

# --- 3. TEXT PREPROCESSING FUNCTION ---
def clean_text(text: str) -> str:
    """
    Membersihkan teks dari URL, mention, hashtag, angka, dan spasi berlebih,
    namun secara spesifik tetap mempertahankan emoji.
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = re.sub(r'http\S+|www.\S+', '', text)  # Hapus URL
    text = re.sub(r'@\w+|#\w+', '', text)        # Hapus mention & hashtag
    text = re.sub(r'\d+', '', text)              # Hapus angka
    text = re.sub(r'\s+', ' ', text).strip()     # Hapus spasi berlebih
    
    return text

# --- 4. DATA TRANSFORMATION & CLEANING ---
# Terapkan fungsi pembersihan pada kolom teks utama
df['text'] = df['text'].apply(clean_text)

# Bersihkan kolom 'ownerUsername' (hanya simpan karakter alfanumerik, _, .)
df['ownerUsername'] = df['ownerUsername'].astype(str).apply(
    lambda x: re.sub(r'[^A-Za-z0-9_.]', '', x)
)

# Konversi dan pastikan kolom 'likesCount' berupa tipe data integer murni
df['likesCount'] = pd.to_numeric(df['likesCount'], errors='coerce').fillna(0).astype(int)

# Bersihkan kolom URL dari spasi putih atau karakter tersembunyi
df['postUrl'] = df['postUrl'].astype(str).apply(lambda x: x.strip())
df['commentUrl'] = df['commentUrl'].astype(str).apply(lambda x: x.strip())

# Bersihkan kolom 'source_file' (hanya simpan karakter alfanumerik, _, ., -)
df['source_file'] = df['source_file'].astype(str).apply(
    lambda x: re.sub(r'[^A-Za-z0-9_.-]', '', x)
)

# Standardisasi format timestamp ke datetime dan hapus data waktu yang invalid (NaT)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp'])

# Filter akhir: Hapus baris di mana kolom teks menjadi benar-benar kosong setelah dibersihkan
df = df[df['text'].str.strip() != ""]

# --- 5. EXPORT CLEANED DATA ---
print(f"Jumlah data setelah cleaning: {df.shape}")

# Menyimpan hasil akhir ke file CSV baru (disesuaikan nama string agar sinkron dengan nama file)
# df.to_csv('cleaned_data.csv', index=False)
# print("✅ Data cleaned berhasil disimpan ke 'cleaned_data.csv' (emoji tetap dipertahankan)")

Jumlah data sebelum cleaning: (80163, 8)
Setelah hapus duplikat: (80163, 8)
Jumlah data setelah cleaning: (80163, 8)
